# Día 4 — Clustering y segmentación económica

Explorar regímenes país-año con K-Means y clustering jerárquico.

## Objetivos de aprendizaje

- Preparar y escalar indicadores comparables.
- Elegir y justificar K con codo y Silhouette.
- Leer centroides, tamaños y dendrogramas.
- Comunicar patrones como exploratorios, no causales ni oficiales.

> **Tiempo del plan oficial:** 3 horas sincrónicas + 1 hora autónoma. Prioriza selección de K, perfiles y dendrograma; realiza una prueba de estabilidad adicional de forma autónoma.

**Dataset:** panel local de indicadores anuales del Banco Mundial en `datos/world_bank_latam_ml.csv`. La ejecución no consulta APIs.

## Cómo estudiar esta jornada

Lee primero la pregunta y la explicación; ejecuta luego las celdas en orden. Después de cada tabla o gráfico, detente en **Interpreta** y redacta una conclusión con evidencia. Los resultados dependen de la versión de las librerías y de las particiones; usa tus propios números y no copies los ejemplos como verdades universales. No confundas predicción con causalidad.

## 1. Pregunta de análisis y unidad de observación

¿Se observan **regímenes macroeconómicos** parecidos en América Latina según crecimiento, inflación, desempleo, comercio, inversión y crecimiento poblacional? Usaremos las observaciones país-año. Un cluster describe similitud bajo estas variables; no es una clasificación oficial de países ni una causalidad.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def buscar_datos():
    candidatos = [Path("../datos"), Path("datos"), Path.cwd() / "datos", Path.cwd().parent / "datos"]
    for ruta in candidatos:
        if ruta.exists():
            return ruta.resolve()
    raise FileNotFoundError("No encuentro la carpeta datos/. Abre Jupyter desde la raíz o notebooks/ del repositorio.")

DATA_DIR = buscar_datos()
print("Datos locales:", DATA_DIR)
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import linkage, dendrogram

archivo = DATA_DIR / "world_bank_latam_ml.csv"
macro = pd.read_csv(archivo)
macro.head()


### Diccionario mínimo y procedencia

El archivo local contiene indicadores anuales descargados del Banco Mundial y 7 países latinoamericanos. Cada fila es un país en un año; puede haber valores faltantes porque la cobertura de ciertos indicadores difiere entre países. `iso3`, `pais` y `anio` identifican observaciones, pero no los introducimos como coordenadas de similitud. Los valores de países-años distintos no son independientes: esto limita cualquier generalización.

In [ ]:
features = ["crecimiento_pib_pct", "inflacion_pct", "desempleo_pct",
            "exportaciones_pct_pib", "inversion_pct_pib", "crecimiento_poblacion_pct"]
print("Periodo:", macro.anio.min(), "–", macro.anio.max())
print("Países:", sorted(macro.pais.unique()))
print("Nulos por indicador:")
print(macro[features].isna().sum())
print("Resumen:")
print(macro[features].describe().round(2).T)


### Decisión de calidad

Las escalas son distintas y existen nulos. Para una práctica introductoria haremos análisis de casos completos: conservamos filas con los seis indicadores disponibles. Es transparente y simple, aunque reduce la muestra y puede sesgarla si faltan datos de forma sistemática. En un análisis aplicado investigaríamos mecanismos de ausencia, unidades, revisiones estadísticas y comparabilidad de definiciones.

In [ ]:
analisis = macro.dropna(subset=features).copy().reset_index(drop=True)
X = analisis[features]
scaler = StandardScaler()
Xz = scaler.fit_transform(X)
print("Filas utilizables:", len(analisis), "de", len(macro))
print("Unidad de análisis: país-año; países:", analisis.pais.nunique())
print(pd.DataFrame(Xz, columns=features).agg(["mean", "std"]).round(2))


## 2. K-Means y elección del número de grupos

K-Means asigna observaciones al centroide más cercano y busca reducir la suma de distancias cuadráticas dentro de los grupos. Requiere escalado porque utiliza distancias euclidianas. El codo muestra la reducción de inercia; Silhouette compara cohesión interna y separación. Ninguno decide por nosotros cuál segmentación es útil.

In [ ]:
ks = range(2, 7)
medidas = []
for k in ks:
    m = KMeans(n_clusters=k, n_init=20, random_state=42).fit(Xz)
    medidas.append({"k": k, "inercia": m.inertia_, "silhouette": silhouette_score(Xz, m.labels_)})
tabla_k = pd.DataFrame(medidas)
tabla_k.round(3)


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(tabla_k.k, tabla_k.inercia, marker="o"); ax[0].set(title="Método del codo", xlabel="k", ylabel="Inercia")
ax[1].plot(tabla_k.k, tabla_k.silhouette, marker="o"); ax[1].set(title="Silhouette", xlabel="k", ylabel="Promedio")
plt.tight_layout()


### Interpreta codo y Silhouette

La inercia siempre disminuye al crecer `k`; no es evidencia de que muchos grupos sean mejores. Busca rendimientos decrecientes en el codo y valores de Silhouette relativamente altos, pero considera que hay solo siete países y observaciones repetidas por año. Selecciona `k` por legibilidad, estabilidad y sentido económico; registra por qué, no solo el máximo de una métrica.

## 3. Ajustar e interpretar perfiles

Usaremos `k=3` como decisión didáctica inicial: permite comparar pocos perfiles y no afirma que existan tres regímenes naturales. Convertiremos centroides a unidades originales para que las conclusiones sean legibles.

In [ ]:
k_elegido = 3
kmeans = KMeans(n_clusters=k_elegido, n_init=30, random_state=42)
analisis["cluster"] = kmeans.fit_predict(Xz)
centros = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=features)
centros.index.name = "cluster"
print("Tamaño de grupos:")
print(analisis.cluster.value_counts().sort_index())
print("Perfiles (centroides en unidades originales):")
print(centros.round(2))


### Traduce perfiles sin inventar etiquetas

Compara cada centroide con la media global y los rangos de `describe()`. Nombra los grupos con descriptores observables (por ejemplo, «mayor crecimiento / mayor inversión») y evita etiquetas normativas como «buenos» o «malos». Los IDs 0, 1, 2 son arbitrarios y pueden permutarse al reentrenar. La media de un cluster no describe todos sus integrantes.

### 3.1 Detección de patrones por país y periodo

Además de los centroides, revisamos dónde y cuándo aparecen las asignaciones. Una concentración temporal puede indicar un ciclo común o un cambio de régimen; no convierte la etiqueta en pronóstico ni permite atribuir causas. Como un país aporta varios años, lee proporciones y tamaños antes de comparar países.

In [ ]:
print('Distribución de clusters dentro de cada país:')
print(pd.crosstab(analisis.pais, analisis.cluster, normalize='index').round(2))

analisis['periodo'] = pd.cut(analisis.anio, bins=[2004, 2009, 2014, 2019, 2024],
                            labels=['2005–2009', '2010–2014', '2015–2019', '2020–2024'])
print('Distribución por periodo:')
print(pd.crosstab(analisis.periodo, analisis.cluster, normalize='index').round(2))

**Interpreta el patrón:** ¿algún cluster se concentra en años específicos o aparece en varios países? Contrasta porcentajes con recuentos absolutos: un porcentaje alto basado en pocos años puede ser inestable. El panel es observacional y pequeño; los patrones sirven para formular preguntas de análisis, no para explicar causas macroeconómicas.

In [ ]:
# Proyección descriptiva: dos indicadores seleccionados, NO son el espacio completo de seis dimensiones.
fig, ax = plt.subplots(figsize=(8, 5))
for c, grupo in analisis.groupby("cluster"):
    ax.scatter(grupo["crecimiento_pib_pct"], grupo["inflacion_pct"], label=f"Cluster {c}", alpha=.75)
ax.set(xlabel="Crecimiento del PIB (%)", ylabel="Inflación (%)", title="País-año por cluster: vista parcial")
ax.legend(); ax.grid(alpha=.2)


### Describe la vista

¿Se separan los grupos en crecimiento e inflación? La figura muestra solo dos de seis variables; una superposición en 2D no prueba que los grupos sean idénticos y una separación aparente no garantiza utilidad. Comprueba también los perfiles de todas las variables.

## 4. Clustering jerárquico y dendrograma

El método aglomerativo inicia con una observación por grupo y fusiona grupos sucesivamente. El dendrograma muestra a qué distancia ocurren las fusiones. Ward minimiza el aumento de variación intragrupo; se usa junto con distancia euclidiana sobre variables estandarizadas.

In [ ]:
Z = linkage(Xz, method="ward")
plt.figure(figsize=(13, 5))
dendrogram(Z, labels=(analisis.pais + "-" + analisis.anio.astype(str)).tolist(), leaf_rotation=90, leaf_font_size=6)
plt.title("Dendrograma Ward — país-año (escala estandarizada)")
plt.ylabel("Distancia de fusión"); plt.tight_layout()


### ¿Cómo leer el dendrograma?

Observaciones unidas a baja altura son más parecidas bajo la distancia elegida. Un corte horizontal produce grupos, pero la altura no es una probabilidad ni una significancia estadística. Etiquetas cercanas pueden resultar ilegibles por el número de observaciones; sigue las ramas y repite el ajuste aglomerativo con `n_clusters=3` para comparar perfiles con K-Means.

In [ ]:
jer = AgglomerativeClustering(n_clusters=k_elegido, linkage="ward")
analisis["cluster_jerarquico"] = jer.fit_predict(Xz)
print(pd.crosstab(analisis.cluster, analisis.cluster_jerarquico, normalize="index").round(2))


### Comparar métodos y estabilidad

La tabla cruzada compara asignaciones, pero los números de cluster de métodos distintos no tienen correspondencia semántica automática. Las etiquetas pueden permutarse. Una comparación formal usaría métricas de concordancia como adjusted Rand index y, sobre todo, perfiles sustantivos. Prueba otra semilla o periodo: si cambian los perfiles, presenta la segmentación como frágil.

## 5. Taller de segmentación económica (20 puntos sugeridos)

1. Documenta unidad de observación, periodo, fuente y faltantes.
2. Compara `k=2..6` con codo y Silhouette.
3. Elige y justifica `k`; informa tamaños y centroides en escala original.
4. Compara K-Means con Ward y explica diferencias.
5. Escribe dos hallazgos descriptivos y tres limitaciones; no atribuyas causalidad ni conviertas clusters en categorías oficiales de países.

**Salida:** notebook con tablas/gráficos y una síntesis de hasta 200 palabras para un analista económico.